# Comparación de Random Forest y Extra Trees en ASSISTments

Notebook complementario de la práctica 1-2. Compara tres corridas sobre el mismo CSV procesado de 3 000 estudiantes y la misma partición oficial: Random Forest de la práctica, Random Forest con hiperparámetros modificados y Extra Trees, algoritmo incluido en el estudio original [1].

La comparación conserva las 722 variables, las etiquetas score y la división source_split. Los parámetros de la segunda corrida se fijan manualmente; no se ejecuta búsqueda automática. Las métricas de entrenamiento describen ajuste y las de evaluation se calculan sobre los registros reservados.


## 1. Cargar el CSV preparado

Sube `assistments_rf_dataset.csv.gz` desde Colab. Esta notebook compara modelos sobre el mismo archivo preparado; no vuelve a integrar archivos crudos.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.colab import files

files.upload()  # Selecciona assistments_rf_dataset.csv.gz
DATASET_CSV = Path('/content/assistments_rf_dataset.csv.gz')
df = pd.read_csv(DATASET_CSV, low_memory=False)

print(f'Filas: {len(df):,} | estudiantes: {df.student_id.nunique():,} | columnas: {df.shape[1]:,}')
display(df['source_split'].value_counts().rename_axis('partición').to_frame('filas'))
display(df['score'].value_counts().sort_index().rename_axis('score').to_frame('filas'))


## 2. Definir X, y y la partición

`score` es la variable dependiente. Se excluye de `X`, junto con `student_id`, `source_split`, `unit_test_assignment_log_id` y `problem_id`. Se usan las particiones oficiales `train` y `evaluation`; no se crea una partición de respaldo.

La revisión de estudiantes compartidos se mantiene porque limita la interpretación de evaluation como prueba de estudiantes completamente nuevos.


In [ ]:
# score es la etiqueta; los IDs y source_split no son predictores.
id_columns = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
feature_columns = [c for c in df.columns if c not in id_columns]
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
y = df['score'].astype('int8')

# Se usan directamente las particiones oficiales source_split.
train_mask = df['source_split'].eq('train')
eval_mask = df['source_split'].eq('evaluation')
X_train = X.loc[train_mask].reset_index(drop=True)
X_test = X.loc[eval_mask].reset_index(drop=True)
y_train = y.loc[train_mask].to_numpy()
y_test = y.loc[eval_mask].to_numpy()

train_students = set(df.loc[train_mask, 'student_id'].dropna())
eval_students = set(df.loc[eval_mask, 'student_id'].dropna())
overlap = train_students & eval_students
shared_eval_rows = df.loc[eval_mask, 'student_id'].isin(train_students).sum()

print(f'Predictores: {len(feature_columns)} | train: {len(y_train):,} | evaluation: {len(y_test):,}')
print(f'Estudiantes compartidos: {len(overlap)} | registros evaluation de esos estudiantes: {shared_eval_rows:,}')


## 3. Modelos e hiperparámetros

- **RF de la práctica:** 600 árboles; profundidad 19; divide grupos desde 10 registros; mínimo 2 por grupo final.
- **RF modificado:** sigue siendo Random Forest, pero baja profundidad a 12 y sube los mínimos a 20 para dividir y 5 por grupo final. Los árboles quedan más limitados; puede reducir sobreajuste, pero también perder patrones.
- **Extra Trees:** también combina árboles, pero prueba cortes al azar y conserva los que mejor separan los datos. En esta corrida usa bootstrap=True.

Los tres usan los mismos datos, 722 predictores, particiones, 600 árboles y semilla 42. Los cambios se fijaron manualmente; no se buscaron parámetros óptimos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

SEED = 42
N_ESTIMATORS = 600

models = {
    'RF práctica': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles de la práctica
        max_depth=19,              # profundidad máxima de la práctica
        min_samples_split=10,      # el grupo necesita 10 filas para dividirse
        min_samples_leaf=2,        # cada grupo final conserva al menos 2 filas
        max_features='sqrt',       # revisa unas 27 variables por división
        bootstrap=True,            # cada árbol usa una muestra con reemplazo
        random_state=SEED,         # hace reproducible el ajuste
        n_jobs=-1                  # usa todos los núcleos disponibles
    ),
    'RF parámetros modificados': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles controlada
        max_depth=12,              # permite menos decisiones en cadena
        min_samples_split=20,      # necesita 20 filas para poder dividirse
        min_samples_leaf=5,        # cada grupo final conserva al menos 5 filas
        max_features='sqrt',       # conserva la selección de predictores
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=19,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,            # en esta corrida, también usa muestras con reemplazo
        random_state=SEED,
        n_jobs=-1
    )
}


## 4. Entrenar y entender las métricas

- **Accuracy:** de todos los registros, ¿en cuántos acertó? RF base: 7 799 de 11 107 = 0,7022.
- **Precisión:** cuando predice una clase, ¿cuántas veces acierta? La precisión macro promedia score=0 y score=1: (0,6604 + 0,7241) / 2 = 0,6923.
- **Recall:** de los casos que realmente eran de una clase, ¿cuántos encontró? El recall macro es (0,5570 + 0,8023) / 2 = 0,6797. Para score=0 encontró 55,70 %.
- **F1:** combina precisión y recall. F1 macro promedia ambas clases: (0,6043 + 0,7612) / 2 = 0,6828.

El asterisco significa “promedio macro”: cada clase pesa igual. No significa significancia estadística. La tabla de prueba es la referencia para comparar; la de entrenamiento muestra el ajuste sobre datos ya vistos.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, confusion_matrix
)

train_rows = []
evaluation_rows = []
roc_data = {}
confusion_data = {}

for model_name, model in models.items():
    # El modelo aprende solamente de la partición de entrenamiento.
    model.fit(X_train, y_train)
    roc_data[model_name] = {}
    confusion_data[model_name] = {}

    for split_name, table_rows, X_part, y_part in [
        ('train', train_rows, X_train, y_train),
        ('evaluation', evaluation_rows, X_test, y_test)
    ]:
        predicted = model.predict(X_part)
        # Precisión: de lo predicho como clase, qué proporción acertó.
        # Recall: de los casos reales de la clase, cuántos encontró. F1 combina ambos.
        # Las medidas macro se calculan en score=0 y score=1 y se promedian por igual.
        table_rows.append({
            'Algoritmo': model_name,
            'Accuracy': accuracy_score(y_part, predicted),
            'Precisión*': precision_score(
                y_part, predicted, average='macro', zero_division=0
            ),
            'Recall*': recall_score(
                y_part, predicted, average='macro', zero_division=0
            ),
            'F1-score*': f1_score(
                y_part, predicted, average='macro', zero_division=0
            )
        })

        # Los siguientes resultados se usan solo para dibujar las gráficas.
        # La matriz usa y real frente a predicción: filas reales, columnas predichas.
        confusion_data[model_name][split_name] = confusion_matrix(
            y_part, predicted, labels=[0, 1]
        )
        class_index = {int(label): i for i, label in enumerate(model.classes_)}
        probability_score0 = model.predict_proba(X_part)[:, class_index[0]]
        actual_score0 = (y_part == 0).astype('int8')
        # ROC de score=0: compara no éxitos encontrados y falsas alertas por umbral.
        fpr, tpr, _ = roc_curve(actual_score0, probability_score0)
        auc_score0 = roc_auc_score(actual_score0, probability_score0)
        roc_data[model_name][split_name] = (fpr, tpr, auc_score0)

train_metrics = pd.DataFrame(train_rows)
evaluation_metrics = pd.DataFrame(evaluation_rows)

display(Markdown(f'### Tabla: métricas de entrenamiento *(n = {len(y_train):,})*'))
display(train_metrics.round(4))
display(Markdown(f'### Tabla: métricas de evaluación o prueba *(n = {len(y_test):,})*'))
display(evaluation_metrics.round(4))
display(Markdown('**Nota:** el asterisco (`*`) indica promedio macro: se promedian `score=0` y `score=1` con el mismo peso.'))

train_metrics.to_csv('metricas_entrenamiento.csv', index=False, float_format='%.4f')
evaluation_metrics.to_csv('metricas_evaluacion.csv', index=False, float_format='%.4f')
print('Guardados: metricas_entrenamiento.csv y metricas_evaluacion.csv')


## 5. Cómo leer las tablas

Train muestra el ajuste con los registros usados para aprender. Evaluation muestra resultados en registros reservados y sirve para comparar los modelos.

El promedio macro calcula cada medida por separado para score=0 y score=1 y luego saca el promedio simple. Así, una clase no pesa más solo por tener más registros.

La partición separa registros, pero comparte 31 estudiantes y 552 filas de evaluation pertenecen a esos estudiantes. No son necesariamente filas duplicadas. Para medir alumnos completamente nuevos, se debe separar agrupando por student_id.

## 6. Gráfico comparativo de las métricas

Cada panel compara los tres modelos con la misma escala: el de entrenamiento muestra el ajuste; el de prueba muestra el resultado en registros reservados. Cada color es una métrica. Una diferencia grande entre train y prueba indica que el desempeño cambió en datos no usados para aprender.

In [ ]:
# Comparación dinámica de las cuatro métricas acordadas.
# Se usan las tablas calculadas arriba; no se vuelven a entrenar los modelos.
metric_specs = [
    ('Accuracy', 'Accuracy', '#2a6f97'),
    ('Precisión*', 'Precisión macro*', '#ee9b00'),
    ('Recall*', 'Recall macro*', '#2a9d8f'),
    ('F1-score*', 'F1 macro*', '#9b5de5')
]
model_order = list(models.keys())
x = np.arange(len(model_order))
bar_width = 0.19

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), dpi=180, sharey=True)
for ax, metrics_table, panel_title in [
    (axes[0], train_metrics, 'Entrenamiento'),
    (axes[1], evaluation_metrics, 'Evaluación (prueba)')
]:
    indexed = metrics_table.set_index('Algoritmo').loc[model_order]
    for metric_index, (column, label, color) in enumerate(metric_specs):
        offset = (metric_index - (len(metric_specs) - 1) / 2) * bar_width
        bars = ax.bar(
            x + offset, indexed[column].to_numpy(), width=bar_width,
            label=label, color=color
        )
        ax.bar_label(bars, fmt='%.2f', fontsize=7, padding=2)
    ax.set_title(panel_title)
    ax.set_xticks(x, model_order, rotation=8)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Valor de la métrica')
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle('Comparación de las cuatro métricas', y=1.10)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig('comparacion_metricas_train_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: comparacion_metricas_train_evaluation.png')


## 7. Curva ROC y matrices de confusión

**ROC:** X muestra la proporción de falsas alertas; Y muestra la proporción de casos score=0 encontrados. Cada punto usa un umbral distinto. La diagonal equivale al azar; una curva más alta y hacia la izquierda suele separar mejor. El AUC resume esa separación y no es accuracy. El umbral no se eligió usando evaluation.

**Matriz:** filas = resultado real; columnas = predicción. En el orden [[real 0/predicho 0, real 0/predicho 1], [real 1/predicho 0, real 1/predicho 1]], la diagonal son aciertos. RF base: encontró 2 526 no éxitos, omitió 2 009, dio 1 299 falsas alertas y acertó 5 273 éxitos.

En las otras matrices, se leen las celdas con la misma regla: arriba izquierda son no éxitos encontrados, arriba derecha no éxitos omitidos, abajo izquierda falsas alertas y abajo derecha éxitos acertados.

In [ ]:
colors = {
    'RF práctica': '#1f4e79',
    'RF parámetros modificados': '#d28e00',
    'Extra Trees': '#24805a'
}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=180)
for ax, split_name, title in zip(
    axes, ['train', 'evaluation'], ['Entrenamiento', 'Evaluación (prueba)']
):
    for model_name in models:
        fpr, tpr, auc_score0 = roc_data[model_name][split_name]
        ax.plot(
            fpr, tpr, lw=1.8, color=colors[model_name],
            label=f'{model_name} (AUC={auc_score0:.3f})'
        )
    ax.plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Referencia al azar')
    ax.set(
        xlim=(0, 1), ylim=(0, 1.02),
        xlabel='Tasa de falsos positivos',
        ylabel='Tasa de verdaderos positivos', title=title
    )
    ax.grid(alpha=.2)
    ax.legend(fontsize=7, loc='lower right')
fig.suptitle('Curvas ROC para score=0')
fig.tight_layout()
fig.savefig('curvas_ROC_train_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: curvas_ROC_train_evaluation.png')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), dpi=180)
for ax, model_name in zip(axes, models):
    matrix = confusion_data[model_name]['evaluation']
    ax.imshow(matrix, cmap='Blues')
    ax.set_title(model_name, fontsize=9)
    ax.set_xticks([0, 1], ['Predicho 0', 'Predicho 1'], fontsize=7)
    ax.set_yticks([0, 1], ['Real 0', 'Real 1'], fontsize=7)
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(
            col, row, f'{value:,}', ha='center', va='center', fontsize=8,
            color='white' if value > matrix.max() * .55 else 'black'
        )
fig.suptitle('Matrices de confusión en evaluación (prueba)')
fig.tight_layout()
fig.savefig('matrices_confusion_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: matrices_confusion_evaluation.png')


## Referencia

[1] A. Yaramala, S. Farokhi y H. Karimi, “Navigating the Data-Rich Landscape of Online Learning: Insights and Predictions from ASSISTments,” Proceedings of the 17th International Conference on Educational Data Mining, pp. 321–331, Atlanta, Georgia, EE. UU., jul. 2024.
